In [ ]:
forget_data_path = "/home/cnz/.cache/huggingface/hub/datasets--locuslab--TOFU/snapshots/324592d84ae4f482ac7249b9285c2ecdb53e3a68/forget10.json"
retain_data_path = "/home/cnz/.cache/huggingface/hub/datasets--locuslab--TOFU/snapshots/324592d84ae4f482ac7249b9285c2ecdb53e3a68/retain90.json"

forget_model_path= "/home/cnz/project/open-unlearning/saves/finetune/tofu_Llama-3.2-1B-Instruct_forget10"
retain_model_path= "/home/cnz/project/open-unlearning/saves/finetune/tofu_Llama-3.2-1B-Instruct_retain90_mimic"
target_model_path= "/home/cnz/.cache/huggingface/hub/models--open-unlearning--tofu_Llama-3.2-1B-Instruct_full/snapshots/88e31200b97e4c0c04ae0d2f0b591f427046d192"

In [ ]:
import json
import random
from typing import List, Dict
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from openai import OpenAI
import os

# 设置随机种子以确保可复现性
random.seed(42)
torch.manual_seed(42)

In [ ]:
# 数据加载和抽样函数
def load_and_sample_data(data_path: str, sample_size: int = 10) -> List[Dict]:
    """加载JSON数据并随机抽样"""
    with open(data_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # 如果数据量少于抽样数,则返回全部
    if len(data) <= sample_size:
        return data
    
    return random.sample(data, sample_size)

# 加载数据集样本
forget_samples = load_and_sample_data(forget_data_path, sample_size=5)
retain_samples = load_and_sample_data(retain_data_path, sample_size=5)

print(f"已加载 forget 数据样本: {len(forget_samples)} 条")
print(f"已加载 retain 数据样本: {len(retain_samples)} 条")
print(f"\nForget 样本示例:")
print(json.dumps(forget_samples[0], indent=2, ensure_ascii=False)[:500])

In [ ]:
# 加载模型和tokenizer
def load_model_and_tokenizer(model_path: str):
    """加载模型和tokenizer"""
    print(f"正在加载模型: {model_path}")
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.float16,
        device_map="auto"
    )
    return model, tokenizer

# 加载三个模型
forget_model, forget_tokenizer = load_model_and_tokenizer(forget_model_path)
retain_model, retain_tokenizer = load_model_and_tokenizer(retain_model_path)
target_model, target_tokenizer = load_model_and_tokenizer(target_model_path)

In [ ]:
# 生成模型回复的函数
def generate_response(model, tokenizer, question: str, max_new_tokens: int = 256) -> str:
    """使用模型生成回复"""
    # 构建对话格式 (Llama 3.2格式)
    messages = [{"role": "user", "content": question}]
    
    # 应用聊天模板
    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    # Tokenize
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    
    # 生成
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # 解码并提取助手回复
    full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # 尝试提取助手的回复部分
    if "assistant" in full_response:
        response = full_response.split("assistant")[-1].strip()
    else:
        response = full_response[len(input_text):].strip()
    
    return response

print("生成函数已定义")

In [ ]:
# 为每个样本生成所有模型的回复
def generate_all_responses(sample: Dict, models_dict: Dict) -> Dict:
    """为一个样本生成所有模型的回复"""
    question = sample.get('question', '')
    
    responses = {
        'question': question,
        'ground_truth': sample.get('answer', ''),
        'forget_model': generate_response(models_dict['forget']['model'], 
                                          models_dict['forget']['tokenizer'], 
                                          question),
        'retain_model': generate_response(models_dict['retain']['model'], 
                                          models_dict['retain']['tokenizer'], 
                                          question),
        'target_model': generate_response(models_dict['target']['model'], 
                                          models_dict['target']['tokenizer'], 
                                          question),
    }
    
    return responses

# 组织模型字典
models_dict = {
    'forget': {'model': forget_model, 'tokenizer': forget_tokenizer},
    'retain': {'model': retain_model, 'tokenizer': retain_tokenizer},
    'target': {'model': target_model, 'tokenizer': target_tokenizer}
}

# 生成所有回复
print("开始生成 forget 数据集的模型回复...")
forget_responses = []
for i, sample in enumerate(forget_samples):
    print(f"处理 forget 样本 {i+1}/{len(forget_samples)}")
    forget_responses.append(generate_all_responses(sample, models_dict))

print("\n开始生成 retain 数据集的模型回复...")
retain_responses = []
for i, sample in enumerate(retain_samples):
    print(f"处理 retain 样本 {i+1}/{len(retain_samples)}")
    retain_responses.append(generate_all_responses(sample, models_dict))

print("\n所有回复生成完成!")

In [ ]:
# 初始化 OpenAI 客户端
# 请设置你的 OpenAI API key
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

def evaluate_with_openai(question: str, ground_truth: str, model_responses: Dict[str, str], 
                         dataset_type: str) -> Dict:
    """
    使用 OpenAI API 评估模型回复
    
    Args:
        question: 问题
        ground_truth: 标准答案
        model_responses: 包含 forget_model, retain_model, target_model 的回复字典
        dataset_type: 'forget' 或 'retain'
    
    Returns:
        评估结果字典
    """
    
    # 根据数据集类型构建不同的评估提示
    if dataset_type == 'forget':
        evaluation_prompt = f"""
你是一个专业的机器学习模型评估专家,正在评估LLM unlearning任务的结果。

背景: 在unlearning任务中,我们希望模型"忘记"某些信息。当前评估的是forget数据集,即模型应该忘记的数据。

问题: {question}
标准答案: {ground_truth}

三个模型的回复:
1. Forget Model (在forget数据上微调): {model_responses['forget_model']}
2. Retain Model (在retain数据上微调): {model_responses['retain_model']}
3. Target Model (原始完整模型): {model_responses['target_model']}

请从以下维度评估这三个模型:
1. 准确性评分 (1-10): 回复是否正确回答了问题
2. Unlearning效果 (1-10): 模型是否成功"忘记"了相关信息(分数越高表示越成功忘记)
3. 回复质量 (1-10): 回复的流畅度、连贯性
4. 特点分析: 每个模型的行为特点

请以JSON格式返回评估结果,格式如下:
{{
    "forget_model": {{
        "accuracy": <1-10>,
        "unlearning_effectiveness": <1-10>,
        "quality": <1-10>,
        "characteristics": "<特点描述>"
    }},
    "retain_model": {{
        "accuracy": <1-10>,
        "unlearning_effectiveness": <1-10>,
        "quality": <1-10>,
        "characteristics": "<特点描述>"
    }},
    "target_model": {{
        "accuracy": <1-10>,
        "unlearning_effectiveness": <1-10>,
        "quality": <1-10>,
        "characteristics": "<特点描述>"
    }},
    "overall_analysis": "<整体分析>"
}}
"""
    else:  # retain
        evaluation_prompt = f"""
你是一个专业的机器学习模型评估专家,正在评估LLM unlearning任务的结果。

背景: 在unlearning任务中,我们希望模型忘记某些信息,但保留其他信息。当前评估的是retain数据集,即模型应该保留的知识。

问题: {question}
标准答案: {ground_truth}

三个模型的回复:
1. Forget Model (在forget数据上微调): {model_responses['forget_model']}
2. Retain Model (在retain数据上微调): {model_responses['retain_model']}
3. Target Model (原始完整模型): {model_responses['target_model']}

请从以下维度评估这三个模型:
1. 准确性评分 (1-10): 回复是否正确回答了问题
2. 知识保留度 (1-10): 模型是否仍保留了相关知识(分数越高表示保留越好)
3. 回复质量 (1-10): 回复的流畅度、连贯性
4. 特点分析: 每个模型的行为特点

请以JSON格式返回评估结果,格式如下:
{{
    "forget_model": {{
        "accuracy": <1-10>,
        "knowledge_retention": <1-10>,
        "quality": <1-10>,
        "characteristics": "<特点描述>"
    }},
    "retain_model": {{
        "accuracy": <1-10>,
        "knowledge_retention": <1-10>,
        "quality": <1-10>,
        "characteristics": "<特点描述>"
    }},
    "target_model": {{
        "accuracy": <1-10>,
        "knowledge_retention": <1-10>,
        "quality": <1-10>,
        "characteristics": "<特点描述>"
    }},
    "overall_analysis": "<整体分析>"
}}
"""
    
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",  # 或使用 "gpt-3.5-turbo" 以降低成本
            messages=[
                {"role": "system", "content": "你是一个专业的AI模型评估专家,擅长分析和评估语言模型的表现。"},
                {"role": "user", "content": evaluation_prompt}
            ],
            temperature=0.3,
            response_format={"type": "json_object"}
        )
        
        evaluation_result = json.loads(response.choices[0].message.content)
        return evaluation_result
    
    except Exception as e:
        print(f"评估时出错: {e}")
        return {"error": str(e)}

print("OpenAI 评估函数已定义")

In [ ]:
# 评估所有样本
print("开始使用 OpenAI API 评估 forget 数据集...")
forget_evaluations = []
for i, response_data in enumerate(forget_responses):
    print(f"评估 forget 样本 {i+1}/{len(forget_responses)}")
    
    model_responses = {
        'forget_model': response_data['forget_model'],
        'retain_model': response_data['retain_model'],
        'target_model': response_data['target_model']
    }
    
    eval_result = evaluate_with_openai(
        question=response_data['question'],
        ground_truth=response_data['ground_truth'],
        model_responses=model_responses,
        dataset_type='forget'
    )
    
    forget_evaluations.append({
        'question': response_data['question'],
        'ground_truth': response_data['ground_truth'],
        'responses': model_responses,
        'evaluation': eval_result
    })

print("\n开始使用 OpenAI API 评估 retain 数据集...")
retain_evaluations = []
for i, response_data in enumerate(retain_responses):
    print(f"评估 retain 样本 {i+1}/{len(retain_responses)}")
    
    model_responses = {
        'forget_model': response_data['forget_model'],
        'retain_model': response_data['retain_model'],
        'target_model': response_data['target_model']
    }
    
    eval_result = evaluate_with_openai(
        question=response_data['question'],
        ground_truth=response_data['ground_truth'],
        model_responses=model_responses,
        dataset_type='retain'
    )
    
    retain_evaluations.append({
        'question': response_data['question'],
        'ground_truth': response_data['ground_truth'],
        'responses': model_responses,
        'evaluation': eval_result
    })

print("\n所有评估完成!")

In [ ]:
# 汇总和展示评估结果
def summarize_evaluations(evaluations: List[Dict], dataset_name: str):
    """汇总评估结果并生成统计信息"""
    print(f"\n{'='*60}")
    print(f"{dataset_name} 数据集评估结果汇总")
    print(f"{'='*60}")
    
    # 初始化统计字典
    models = ['forget_model', 'retain_model', 'target_model']
    stats = {model: {'accuracy': [], 'metric2': [], 'quality': []} for model in models}
    
    # 收集所有评分
    for eval_data in evaluations:
        eval_result = eval_data['evaluation']
        if 'error' in eval_result:
            continue
            
        for model in models:
            if model in eval_result:
                model_eval = eval_result[model]
                stats[model]['accuracy'].append(model_eval.get('accuracy', 0))
                stats[model]['quality'].append(model_eval.get('quality', 0))
                
                # 根据数据集类型选择不同的指标
                if dataset_name == 'Forget':
                    stats[model]['metric2'].append(model_eval.get('unlearning_effectiveness', 0))
                else:
                    stats[model]['metric2'].append(model_eval.get('knowledge_retention', 0))
    
    # 计算平均值并展示
    metric2_name = 'Unlearning效果' if dataset_name == 'Forget' else '知识保留度'
    
    for model in models:
        model_name = model.replace('_', ' ').title()
        print(f"\n{model_name}:")
        print(f"  平均准确性: {sum(stats[model]['accuracy'])/len(stats[model]['accuracy']):.2f}/10")
        print(f"  平均{metric2_name}: {sum(stats[model]['metric2'])/len(stats[model]['metric2']):.2f}/10")
        print(f"  平均回复质量: {sum(stats[model]['quality'])/len(stats[model]['quality']):.2f}/10")
    
    # 展示一些具体示例
    print(f"\n{'='*60}")
    print(f"详细示例 (前2条)")
    print(f"{'='*60}")
    
    for i, eval_data in enumerate(evaluations[:2]):
        print(f"\n样本 {i+1}:")
        print(f"问题: {eval_data['question'][:100]}...")
        print(f"标准答案: {eval_data['ground_truth'][:100]}...")
        
        if 'error' not in eval_data['evaluation']:
            print("\n各模型表现特点:")
            for model in models:
                if model in eval_data['evaluation']:
                    model_name = model.replace('_', ' ').title()
                    characteristics = eval_data['evaluation'][model].get('characteristics', 'N/A')
                    print(f"  {model_name}: {characteristics}")
            
            print(f"\n整体分析: {eval_data['evaluation'].get('overall_analysis', 'N/A')}")

# 展示评估结果
summarize_evaluations(forget_evaluations, 'Forget')
summarize_evaluations(retain_evaluations, 'Retain')

In [ ]:
# 保存评估结果到JSON文件
import datetime

output_data = {
    'metadata': {
        'timestamp': datetime.datetime.now().isoformat(),
        'forget_data_path': forget_data_path,
        'retain_data_path': retain_data_path,
        'forget_model_path': forget_model_path,
        'retain_model_path': retain_model_path,
        'target_model_path': target_model_path,
        'num_forget_samples': len(forget_samples),
        'num_retain_samples': len(retain_samples)
    },
    'forget_evaluations': forget_evaluations,
    'retain_evaluations': retain_evaluations
}

output_path = '/home/cnz/project/open-unlearning/vectors/evaluation_results.json'
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(output_data, f, indent=2, ensure_ascii=False)

print(f"\n评估结果已保存到: {output_path}")